### Setup

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")

MODEL = "gpt-4.1-mini"

In [ ]:
# Absolute path is important: the server needs a full path for its allowed directory.
SANDBOX_DIR = os.path.abspath("secret_project_007")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Drop in a couple of sample files for the agent to find.
# encoding="utf-8" keeps the em dashes (—) safe on Windows (which defaults to cp1252).
with open(os.path.join(SANDBOX_DIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Project Falcon — kickoff notes\n"
        "- Estimated budget: $42,000\n"
        "- Launch target: end of Q3\n"
        "- Owner: Priya\n"
        "- Core team: Marco (engineering), Lena (design), Sam (go-to-market)\n"
        "What it is:\n"
        "Falcon is a tech news and discussion community with AI avatars who provide commentary\n"
        "It's like a Hacker News + a Late Night Talk Show, but in written format and with AI commentators.\n"
        "The bet: engaging AI avatars will quickly become fan-favourites because of the unique dynamic between them.\n"
        "Premise:\n"
        "Ship before the competition announces theirs; quiet until then\n"
        "Hacker News is the benchmark — we study it constantly to understand what the community wants\n"
        "Two competitors rumored to be building something similar — speed is the moat\n"
        "'Falcon' is a placeholder; marketing hates it, but it has stuck\n"
        "Priya's one hard rule: do not slip the Q3 date to add features\n"
        "Open question: what are the most popular categories on Hacker News?\n"
    )

with open(os.path.join(SANDBOX_DIR, "todo.md"), "w", encoding="utf-8") as f:
    f.write(
        "# To do\n"
        "- [X] Book venue\n"
        "- [ ] Confirm budget with finance\n"
        "- [X] Lock the core team\n"
        "- [X] Brief the team on keeping this quiet\n"
        "- [ ] Draft announcement\n"
        "- [ ] Decide whether to keep the 'Falcon' name or rebrand before launch\n"
        "- [ ] Pressure-test the Q3 timeline with engineering\n"
        "- [ ] Scan the Hacker News front page, group top 20 stories into broad categories\n"
    )

print(f"✅ Sandbox ready at: {SANDBOX_DIR}")
print("   Files:", os.listdir(SANDBOX_DIR))

#==================================================

# Create a decoy folder
OTHER_DIR = os.path.abspath("secret_project_006")
os.makedirs(OTHER_DIR, exist_ok=True)

# Feels empty.. Drop in a decoy file.
with open(os.path.join(OTHER_DIR, "decoy_file.txt"), "w") as f:
    f.write("This is a decoy file")

print(f"✅ Decoy folder is ready at: {OTHER_DIR}")
print("   Files:", os.listdir(OTHER_DIR))

### Check that Node.js + npx are available

In [ ]:
# If this errors, install Node from https://nodejs.org/ and ensure it's in your PATH.
!node --version
!npx --version

## Filesystem Fetch Server

In [ ]:
WEB_AGENT_PROMPT = """
You are a helpful assistant.
When the user asks about a web page, if you can access the internet, then read the page first, then answer based on what you actually read.
Always cite the URL you read from in the format [source: <URL>].
If you cannot access the internet, then say so.
"""

In [ ]:
fetch_server_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"]
}

In [ ]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server1:
    tools = await server1.list_tools()

    print(f"✅ Connected. The server offers {len(tools)} tools(s):\n")
    for tool in tools:
        print(f"🔧 - {tool.name}: {tool.description}")

## Filesystem MCP Server

In [ ]:
filesystem_server_params = {
    "command": "npx",
    "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        SANDBOX_DIR
    ]
}

In [ ]:
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    fs_tools = await server2.list_tools()

    print(f"✅ MCP Server connected. Server offers {len(fs_tools)} tools:\n")
    for tool in fs_tools:
        print(f"🔧 - {tool.name}: {tool.description.strip().splitlines()[0]}")
    

## Filesystem Agent (with MCP Server)

In [ ]:
FILES_AGENT_PROMPT = f"""
You are a file assistant. You work inside the directory {SANDBOX_DIR}.
Always use FULL paths under {SANDBOX_DIR} (e.g. {SANDBOX_DIR}/notes.txt).
You have filesystem tools (via an MCP server) to list, read, search, write, and edit files.
When asked about files, first list or read what's there, then act based on what you actually
find. Be concise, and tell the user exactly which files you read or changed.
"""

In [ ]:
# Read Files
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"What files are in my folder, and what is each one about? Give me a one-line summary per file.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

In [ ]:
# Search files for a keyword
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"Search for every file that mentions Budget (in the file contents)",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

In [ ]:
# Test MCP Server Side Boundaries
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"Please read the folder {OTHER_DIR} and show me what files are there.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

### MCP -> Create a File

In [ ]:
# Create a summary.md file with a summary of the contents of the other files. Then read that file and show me the summary.
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"Create a summary.md file with a summary of the contents of the other files. The file summary.md combines the key facts and outstanding todos into a short project status update. Then read that file and show me the summary.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

## Using Multiple MCP Servers

### CHALLENGE: Two servers, one agent

#### Given a single agent both the Fetch server and the Filesystem server. Then ask it to fetch a specific web page (your choice) and save a summary of it into the sandfox folder as a new .md file

In [ ]:
# Instructions for agent to search a web page and write a summary to a file.
WEB_SUMMARY_AGENT_PROMPT = WEB_AGENT_PROMPT + FILES_AGENT_PROMPT + """
When the user asks you to summarize a web page, first read the web page using your fetch server tools, then write a concise summary to an md file (name is given by user) using 
your filesystem server tools, then read back
the file you just wrote and show me the summary.
"""

In [ ]:
#Launch both Fetch server and Filesystem server, and use the Fetch server to get some content from the web, then save it to a file using the Filesystem server.
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server1, \
           MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:

    web_summary_agent = Agent(
        name="Web Summary Agent",
        instructions=WEB_SUMMARY_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server1, server2]
    )

    result = await Runner.run(
        web_summary_agent,
        input=f"Summarize the webpage at https://lite.cnn.com/2026/07/05/us/july-4-shooting-coney-island and save the summary to a file named news.md, then read back the file and show me the summary.",
        max_turns=30
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

In [ ]:
SUMMARIZER_AGENT_PROMPT = f"""Your are a research assistant. You work inside directory {SANDBOX_DIR}.
You have two MCP servers at your disposal: filesystem (list/read/search/write files etc) and fetch (read web pages).
If user asks for information, use fetch to get that page. Summarize it and save it as an md file. Then read back the file and show me the summary.
"""

In [ ]:
# Errors to debug:
# 1. The Agent doesn't read the file.

SUMMARIZER_AGENT_PROMPT = f"""Your are a research assistant. You work inside directory {SANDBOX_DIR}.
You have two MCP servers at your disposal: filesystem (list/read/search/write files etc) and fetch (read web pages).

Always start by listing the directory and reading every local file that could be relevant - don't assume the user named all the files that matter.

If user asks for information, use fetch to get that page. Summarize it and save it as an md file. Then read back the file and show me the summary.
"""

In [ ]:
# Errors to debug:
# 1. The Agent doesn't read the file.
# 2. The Agent announces what it will do and stops.

SUMMARIZER_AGENT_PROMPT = f"""Your are a research assistant. You work inside directory {SANDBOX_DIR}.
You have two MCP servers at your disposal: filesystem (list/read/search/write files etc) and fetch (read web pages).

Always start by listing the directory and reading every local file that could be relevant - don't assume the user named all the files that matter.

If user asks for information, use fetch to get that page. Summarize it and save it as an md file. Then read back the file and show me the summary.

Actually do the work at every step - never just describe what you plan to do and stop.
Recheck the user instructions - you are only done when the requirements are fully satisfied.
Your final message should be you reporting back to confirm that the instructions have been fully fulfilled.
"""

In [ ]:
# Errors to debug:
# 1. The Agent doesn't read the file.
# 2. The Agent announces what it will do and stops.
# 3. The Agent creates a teardown but doesn't save it to a file.

SUMMARIZER_AGENT_PROMPT = f"""Your are a research assistant. You work inside directory {SANDBOX_DIR}.
You have two MCP servers at your disposal: filesystem (list/read/search/write files etc) and fetch (read web pages).

Always start by listing the directory and reading every local file that could be relevant - don't assume the user named all the files that matter.

If user asks for information, use fetch to get that page. Summarize it and save it as an md file. Then read back the file and show me the summary.

Actually do the work at every step - never just describe what you plan to do and stop.
Recheck the user instructions - you are only done when the requirements are fully satisfied.
Your final message should be you reporting back to confirm that the instructions have been fully fulfilled.

CRITICAL: If the user asks you to write a file, do NOT ask the user's permission to write that file.
Proceed as instructed, saving the file is MANDATORY.
"""

In [ ]:
# Final
SUMMARIZER_AGENT_PROMPT = f"""Your are a research assistant. You work inside directory {SANDBOX_DIR}.
You have two MCP servers at your disposal: filesystem (list/read/search/write files etc) and fetch (read web pages).

Always start by listing the directory and reading every local file that could be relevant - don't assume the user named all the files that matter.

If user asks for information, use fetch to get that page. Summarize it and save it as an md file. Then read back the file and show me the summary.

Actually do the work at every step - never just describe what you plan to do and stop.
Recheck the user instructions - you are only done when the requirements are fully satisfied.
Your final message should be you reporting back to confirm that the instructions have been fully fulfilled.

CRITICAL: If the user asks you to write a file, do NOT ask the user's permission to write that file.
Proceed as instructed, saving the file is MANDATORY.
"""

In [ ]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as fetch_server:
    async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as filesystem_server:
        summarizer_agent = Agent(
            name="Summarizer Agent",
            instructions=SUMMARIZER_AGENT_PROMPT,
            model=MODEL,
            mcp_servers=[fetch_server, filesystem_server]
        )

        with trace("Summarizer Agent Run"):
            result = await Runner.run(
                summarizer_agent,
                input=f"Read the Project Falcon notes to understand what we are building. Then fetch " \
                    "https://news.ycombinator.com/ and complete the oustanding research todo item. " \
                    "Write your findings in a competitive teardown in the project folder as teardown.md, then read back the file and show me the summary.",
                max_turns=30
            )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)


In [ ]:
# Errors to debug:
# 1. The Agent doesn't read the file.
# 2. The Agent announces what it will do and stops.
# 3. The Agent creates a teardown but doesn't save it to a file.